In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Acessa a página do blog
url = 'https://www.sicredi.com.br/site/blog/'
resposta = requests.get(url, timeout=30)
resposta.raise_for_status()

# Converte o HTML em um objeto BeautifulSoup
soup = BeautifulSoup(resposta.content, 'html.parser')

# Seleciona os cards das notícias
cards = soup.find_all('div', class_='post-list-card-content')

cards_armazenados = []

for card in cards:
    # Extrai a categoria da notícia
    tag = card.find('span')
    tag = tag.get_text(strip=True) if tag else None

    # Extrai o tempo de leitura ou a data da notícia
    aux = card.find('aside') or card.find('span', class_='post-date')
    aux = aux.get_text(strip=True) if aux else None

    tempo_leitura = None
    data_da_noticia = None

    if aux:
        if 'leitura' in aux.lower():
            tempo_leitura = aux
        else:
            data_da_noticia = aux

    # Extrai o título e o resumo
    titulo = card.select_one('h3.titulo-tamanho-5')
    titulo = titulo.get_text(strip=True) if titulo else None

    texto = card.select_one('p.subtitle')
    texto = texto.get_text(strip=True) if texto else None

    # Procura o link dentro do card ou em uma tag <a> ancestral
    link = card.find('a') or card.find_parent('a')
    apontamento = urljoin(url, link.get('href')) if link and link.get('href') else None

    # Armazena os dados extraídos
    cards_armazenados.append({
        'tag': tag,
        'tempo': tempo_leitura,
        'data_da_noticia': data_da_noticia,
        'titulo': titulo,
        'texto': texto,
        'apontamento': apontamento
    })

# Cria o DataFrame e salva os dados em JSON
df = pd.DataFrame(cards_armazenados)

df.to_json(
    '2026-10-27-cards.json',
    orient='records',
    force_ascii=False,
    indent=2
)

In [2]:
df.head()

,tag,tempo,data_da_noticia,titulo,texto,apontamento
0,Produtos,14 min de leitura,NaN,Como conseguir um cartão de crédito com limite...,Ter um cartão de crédito com limite alto depen...,https://www.sicredi.com.br/site/blog/produtos/...
1,Artigo,12 min de leitura,NaN,Como fazer um cartão de crédito? Passo a passo...,Fazer um cartão de crédito é mais simples do q...,https://www.sicredi.com.br/site/blog/artigo/co...
2,Segurança,7 min de leitura,NaN,Quais são os golpes mais comuns? Conheça 11 di...,"Conhecendo sobre os tipos de golpe, você se se...",https://www.sicredi.com.br/site/blog/seguranca...
3,Segurança,6 min de leitura,NaN,Origem Verificada em ligações: proteja-se de g...,Conte com essa nova ferramenta para ajudar voc...,https://www.sicredi.com.br/site/blog/seguranca...
4,Segurança,8 min de leitura,NaN,Golpe do resgate de pontos: não seja a próxima...,Saiba tudo sobre o golpe do resgate de pontos ...,https://www.sicredi.com.br/site/blog/seguranca...
